# Results

Every number, table and figure in the report, computed from the outputs of `scripts/run_experiments.sh`. Figures are written to `report/figures/`.

## Setup

In [1]:
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from matplotlib.collections import LineCollection
from matplotlib.patches import Rectangle

ROOT = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
os.chdir(ROOT)

from scripts.error_budget import focal, pixels_on_target
from scripts.simulate_triangulation import SCENARIOS, simulate
from ultralytics import YOLO

from dronetrack import datasets
from dronetrack.config import load_config
from dronetrack.experiments import load_metrics
from dronetrack.training_data import box_from_point

config = load_config(Path("configs"))
RUNS, FIGURES = Path("runs"), Path("report/figures")
FIGURES.mkdir(parents=True, exist_ok=True)

In [2]:
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["cmr10"],
    "mathtext.fontset": "cm",
    "axes.formatter.use_mathtext": True,
    "axes.unicode_minus": False,
    "font.size": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.6,
    "legend.frameon": False,
    "savefig.bbox": "tight",
})
BLUE, ORANGE, GREEN, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#d9d8d4"

In [3]:
METHODS = pd.DataFrame(
    [
        ("motion-kf", "comparison", "Motion + Bernoulli"),
        ("yolo-bytetrack", "comparison", "Frame YOLO + ByteTrack"),
        ("dronetrack", "comparison", "Crop YOLO + Bernoulli"),
        ("dronetrack-no-motion", "ablation", "Crop YOLO + Bernoulli, no motion crops"),
        ("dronetrack-counter", "ablation", "Crop YOLO + counter"),
        ("dronetrack-cenek-weights", "ablation", "Crop YOLO (base only) + Bernoulli"),
    ],
    columns=["method", "suite", "name"],
).set_index("method")
NAMES = METHODS.name.to_dict()
DETECTORS = {"yolo26s-mixed": "mixed", "yolo26s-cenek": "base only"}


def per_sequence(method, test_set, condition="clean", suite=None):
    suite = suite or METHODS.suite[method]
    return pd.read_csv(RUNS / suite / method / test_set / condition / "per_sequence.csv")


def outputs(method, sequence, test_set="cenek-test"):
    """Per-frame outputs of a method on one sequence"""
    path = RUNS / METHODS.suite[method] / method / test_set / "clean" / "sequences" / f"{sequence.replace('/', '_')}.parquet"
    return pd.read_parquet(path).set_index("frame")


def with_labels(frames, sequence):
    """Per-frame outputs joined with the labels, and whether each frame is a hit within 10 px"""
    labels = sequence.annotations.set_index("frame")[["x", "y", "visible"]].add_suffix("_label")
    joined = frames.join(labels)
    error = np.hypot(joined.x - joined.x_label, joined.y - joined.y_label)
    return joined.assign(hit=joined.declared & joined.visible_label & (error <= 10))

## 1. Design analysis

### 1.1 Pixels on target and coverage

In [4]:
pixels_on_target()

,50 m,100 m,200 m,500 m,1000 m,2000 m
HFOV 90 deg,6.7,3.4,1.7,0.7,0.3,0.2
HFOV 60 deg,11.6,5.8,2.9,1.2,0.6,0.3
HFOV 20 deg,38.1,19.1,9.5,3.8,1.9,1.0
HFOV 5 deg,153.9,77.0,38.5,15.4,7.7,3.8
HFOV 2 deg,385.0,192.5,96.2,38.5,19.2,9.6


In [5]:
def band(hfov, width=3840, drone=0.35, pixels=6, overlap=0.1):
    """Range at which a drone spans `pixels`, and 16:9 cameras needed for a 360 x 30 degree band"""
    vfov = 2 * np.degrees(np.arctan(np.tan(np.radians(hfov) / 2) * 9 / 16))
    columns = np.ceil(360 / (hfov * (1 - overlap)))
    rows = np.ceil(30 / (vfov * (1 - overlap)))
    return {"hfov_deg": hfov, "range_m": focal(hfov, width) * drone / pixels, "cameras": int(columns * rows)}


pd.DataFrame([band(h) for h in (90, 60, 40, 20, 10)]).round(0)

,hfov_deg,range_m,cameras
0,90,112.0,5
1,60,194.0,7
2,40,308.0,20
3,20,635.0,60
4,10,1280.0,240


### 1.2 Two-station triangulation

In [6]:
ranges = [100, 300, 500, 1000, 1500]
triangulation = pd.DataFrame(
    {name: [simulate(z, **params) for z in ranges] for name, params in SCENARIOS.items()}, index=ranges
)
triangulation["closed form, B=300 m"] = [z**2 * 0.2e-3 / 300 for z in ranges]
triangulation.round(2)

,"B=100 m, 0.2 mrad","B=300 m, 0.2 mrad","B=300 m, 0.2 mrad + 0.1 mrad calib.","B=300 m, 0.2 mrad + 5 ms sync","closed form, B=300 m"
100,0.07,0.09,0.10,0.11,0.01
300,0.34,0.20,0.22,0.21,0.06
500,0.88,0.39,0.43,0.40,0.17
1000,3.43,1.24,1.38,1.25,0.67
1500,7.67,2.65,2.95,2.66,1.50


In [7]:
stereo_error = 1000**2 * 0.25 / focal(5)
zoom_mrad = 1e3 / focal(2)
zoom_width = 2 * 1000 * np.tan(np.radians(1))
crossing_rate = np.degrees(30 / 300)

print(f"stereo rig, 5 deg lens, B = 1 m, 0.25 px disparity noise: {stereo_error:.1f} m at 1 km")
print(f"2 deg zoom over 1920 px: {zoom_mrad:.3f} mrad per pixel, {zoom_width:.0f} m wide at 1 km")
print(f"crossing at 30 m/s and 300 m: {crossing_rate:.1f} deg/s, {crossing_rate * 0.1:.2f} deg in 100 ms")
print(f"10 ms clock offset at 30 m/s: {30 * 0.01:.1f} m")

stereo rig, 5 deg lens, B = 1 m, 0.25 px disparity noise: 11.4 m at 1 km
2 deg zoom over 1920 px: 0.018 mrad per pixel, 35 m wide at 1 km
crossing at 30 m/s and 300 m: 5.7 deg/s, 0.57 deg in 100 ms
10 ms clock offset at 30 m/s: 0.3 m


## 2. Data

In [8]:
audit = pd.read_csv("data/cache/audit/summary.csv")
audit.groupby(["dataset", "split"]).agg(
    sequences=("sequence", "size"),
    frames=("video_frames", "sum"),
    visible=("drone_visible_frames", "sum"),
)

sequences  frames  visible
dataset  split                            
cenek    test           7  134043    66078
         train         10  154548    96108
         val            4   22146    18128
dut      test           1    2200     2200
         train          1    5200     5197
         val            1    2600     2600
fred     test          11   37236    30817
         train         40   13336    12010
         val            4    1357     1228
halmstad test          41    4287     1764
         train        191   19942     8014
         val           41    4241     1744

In [9]:
drone_free = per_sequence("dronetrack", "halmstad-negatives")
robustness_frames = audit.set_index("sequence").loc[["dataset4/cam1", "dataset4/cam2", "dataset4/cam6"], "video_frames"]

print(f"Halmstad drone-free test clips: {len(drone_free)}, {drone_free.minutes.sum():.2f} min")
print(f"frames in the robustness subset: {robustness_frames.sum()}")

Halmstad drone-free test clips: 24, 4.15 min
frames in the robustness subset: 42379


In [10]:
tiles = {DETECTORS[m]: pd.read_json(f"data/cache/yolo/{m}/summary.json", typ="series") for m in DETECTORS}
segmented, fallback = tiles["mixed"].cenek_segmented, tiles["mixed"].cenek_fallback

print("training tiles:", {name: t["tiles"]["train"] for name, t in tiles.items()})
print("validation images:", {name: t["tiles"]["val"] for name, t in tiles.items()})
print(f"point-to-box segmentation: {segmented} of {segmented + fallback} ({segmented / (segmented + fallback):.1%})")

training tiles: {'mixed': 26710, 'base only': 11556}
validation images: {'mixed': 9444, 'base only': 2162}
point-to-box segmentation: 11088 of 11440 (96.9%)


In [11]:
sizes = []
for path in Path("data/cache/yolo/yolo26s-cenek/labels/train").glob("*.txt"):
    for line in path.read_text().splitlines():
        _, _, _, w, h = map(float, line.split())
        sizes.append(640 * max(w, h))

print(f"derived training boxes, longer side: median {np.median(sizes):.0f} px (includes 4 px of padding)")

derived training boxes, longer side: median 16 px (includes 4 px of padding)


## 3. Detectors

In [12]:
def training_summary(model):
    log = pd.read_csv(RUNS / "train" / model / "results.csv")
    fitness = 0.1 * log["metrics/mAP50(B)"] + 0.9 * log["metrics/mAP50-95(B)"]
    best = log.loc[fitness.idxmax()]
    return {
        "epochs": len(log),
        "best epoch": int(best.epoch),
        "val AP50 at best": best["metrics/mAP50(B)"],
        "hours": log.time.iloc[-1] / 3600,
    }


pd.DataFrame({DETECTORS[m]: training_summary(m) for m in DETECTORS}).T

,epochs,best epoch,val AP50 at best,hours
mixed,28.0,18.0,0.811,10.435
base only,30.0,30.0,0.968,4.484


In [13]:
detection = pd.DataFrame([pd.read_json(p, typ="series") for p in sorted((RUNS / "detection").glob("*/*.json"))])
detection["model"] = detection.model.map(DETECTORS)
detection.pivot(index="model", columns="dataset", values=["ap50", "ap50_95", "precision", "recall"])

ap50          ap50_95          precision          recall         
dataset      dut halmstad     dut halmstad       dut halmstad    dut halmstad
model                                                                        
base only  0.346    0.016   0.140    0.003     0.819    0.123  0.330    0.096
mixed      0.946    0.898   0.656    0.374     0.954    0.888  0.888    0.897

## 4. Tracker parameters

In [14]:
estimates = {}
for name in ["motion", "yolo26s-mixed", "yolo26s-cenek"]:
    values = yaml.safe_load(open(RUNS / f"estimate/{name}.yaml"))
    counts = yaml.safe_load(open(RUNS / f"estimate/{name}.counts.yaml"))
    estimates[name] = values | {"hits": counts["hits"], "frames": counts["frames"]}

estimates = pd.DataFrame(estimates).T
estimates

,p_detect,clutter_density,measurement_std,process_noise,hits,frames
motion,0.868,9.053e-07,1.000,829.029,15729.0,18128.0
yolo26s-mixed,0.990,7.707e-08,1.084,829.029,3627.0,3627.0
yolo26s-cenek,0.990,1.124e-07,1.099,829.029,3627.0,3627.0


In [15]:
p_detect, clutter, sigma = estimates.loc["yolo26s-mixed", ["p_detect", "clutter_density", "measurement_std"]]
gate, survival = config.pipeline.tracker.gate_probability, config.pipeline.tracker.p_survival

miss = 1 - p_detect * gate  # likelihood ratio of a frame without detection
q = 1.0
for k in (1, 2):
    prior = survival * q
    q = miss * prior / (1 - prior + miss * prior)
    print(f"presence after {k} consecutive miss(es): {q:.2f}")

# a detection at the predicted position of a new track, whose innovation covariance is 2 sigma^2 I
density = 1 / (2 * np.pi * 2 * sigma**2)
print(f"likelihood ratio of one detection: {p_detect * density / clutter:.1e}")

presence after 1 consecutive miss(es): 0.66
presence after 2 consecutive miss(es): 0.04
likelihood ratio of one detection: 8.7e+05


## 5. Tracking

In [16]:
metrics = load_metrics(RUNS)
metrics = metrics[metrics.suite.isin(["comparison", "ablation"]) & (metrics.perturbation == "clean")]
by_test_set = metrics.set_index(["method", "test_set"])

COLUMNS = {
    ("base test", "P@10"): ("cenek-test", "precision_10"),
    ("base test", "P@20"): ("cenek-test", "precision_20"),
    ("base test", "AUC_P"): ("cenek-test", "precision_auc"),
    ("base test", "SA"): ("cenek-test", "state_accuracy"),
    ("FRED", "P@20"): ("fred-test", "precision_20"),
    ("FRED", "AUC_Pn"): ("fred-test", "norm_precision_auc"),
    ("FRED", "AUC_S"): ("fred-test", "success_auc"),
    ("FRED", "SA"): ("fred-test", "state_accuracy"),
    ("Halmstad", "false alarms/min"): ("halmstad-negatives", "false_alarms_per_min"),
    ("base test", "latency p50 (ms)"): ("cenek-test", "latency_p50_ms"),
}
main = pd.DataFrame({
    column: by_test_set.xs(test_set, level="test_set")[metric] for column, (test_set, metric) in COLUMNS.items()
})
main.columns = pd.MultiIndex.from_tuples(main.columns)
main = main.loc[METHODS.index].rename(index=NAMES)
main

base test                        FRED                              Halmstad        base test
                                            P@10   P@20  AUC_P     SA   P@20 AUC_Pn  AUC_S     SA false alarms/min latency p50 (ms)
method                                                                                                                             
Motion + Bernoulli                         0.902  0.908  0.864  0.644  0.612  0.385  0.117  0.197           28.881            4.241
Frame YOLO + ByteTrack                     0.822  0.825  0.794  0.730  0.311  0.228  0.210  0.346            5.054           41.969
Crop YOLO + Bernoulli                      0.794  0.798  0.768  0.733  0.297  0.211  0.196  0.329            3.369           33.868
Crop YOLO + Bernoulli, no motion crops     0.762  0.765  0.737  0.734  0.260  0.186  0.173  0.310            3.369           18.909
Crop YOLO + counter                        0.716  0.721  0.694  0.660  0.329  0.228  0.219  0.346            0.963           35.024
Crop YOLO (base only) + Bernoulli          0.874  0.880  0.847  0.853  0.370  0.233  0.103  0.163           18.532           33.327

In [17]:
cameras = pd.DataFrame({NAMES[m]: per_sequence(m, "cenek-test").set_index("sequence").precision_20 for m in METHODS.index}).T
cameras.columns = [c.split("/")[-1] for c in cameras.columns]
cameras["mean"] = cameras.mean(axis=1)
cameras["mean cam0-5"] = cameras[[f"cam{i}" for i in range(6)]].mean(axis=1)
cameras

,cam0,cam1,cam2,cam3,cam4,cam5,cam6,mean,mean cam0-5
Motion + Bernoulli,0.954,0.975,0.953,0.735,0.833,0.955,0.954,0.908,0.901
Frame YOLO + ByteTrack,0.894,0.830,0.960,0.847,0.809,0.639,0.796,0.825,0.830
Crop YOLO + Bernoulli,0.918,0.924,0.981,0.959,0.796,0.615,0.395,0.798,0.866
"Crop YOLO + Bernoulli, no motion crops",0.907,0.917,0.970,0.842,0.786,0.591,0.346,0.765,0.835
Crop YOLO + counter,0.936,0.739,0.957,0.932,0.810,0.666,0.007,0.721,0.840
Crop YOLO (base only) + Bernoulli,0.950,0.930,0.982,0.907,0.800,0.638,0.951,0.880,0.868


In [18]:
proposed = cameras.loc[NAMES["dronetrack"], [f"cam{i}" for i in range(7)]]
for method in ["yolo-bytetrack", "motion-kf", "dronetrack-no-motion", "dronetrack-counter"]:
    other = cameras.loc[NAMES[method], proposed.index]
    print(f"better than {NAMES[method]} on {(proposed > other).sum()} of 7 cameras")

test_cameras = audit.set_index("sequence").loc[[f"dataset4/cam{i}" for i in range(7)]]
print("drone visible:", (test_cameras.drone_visible_frames / test_cameras.video_frames).round(2).tolist())

better than Frame YOLO + ByteTrack on 4 of 7 cameras
better than Motion + Bernoulli on 2 of 7 cameras
better than Crop YOLO + Bernoulli, no motion crops on 7 of 7 cameras
better than Crop YOLO + counter on 4 of 7 cameras
drone visible: [0.76, 0.27, 0.45, 0.67, 0.52, 0.35, 0.34]


In [19]:
curves = metrics[metrics.test_set == "cenek-test"].set_index("method").precision_curve
for method in ["dronetrack", "motion-kf"]:
    curve = np.asarray(curves[method])
    print(f"{NAMES[method]}: {curve[2] / curve[50]:.0%} of the frames correct within 50 px are within 2 px")

Crop YOLO + Bernoulli: 81% of the frames correct within 50 px are within 2 px
Motion + Bernoulli: 69% of the frames correct within 50 px are within 2 px


In [20]:
alarms = pd.concat([per_sequence(m, "halmstad-negatives").assign(method=NAMES[m]) for m in METHODS.index])
alarms["distractor"] = alarms.sequence.str.split("_").str[1].str.lower()
alarms.pivot_table(
    index="method", columns="distractor", values="false_alarm_onsets", aggfunc="sum", margins=True, margins_name="total"
).loc[NAMES.values()]

distractor,airplane,bird,helicopter,total
method,,,,
Motion + Bernoulli,26,37,57,120
Frame YOLO + ByteTrack,5,0,16,21
Crop YOLO + Bernoulli,4,0,10,14
"Crop YOLO + Bernoulli, no motion crops",4,0,10,14
Crop YOLO + counter,1,0,3,4
Crop YOLO (base only) + Bernoulli,24,24,29,77


In [21]:
points = main[[("Halmstad", "false alarms/min"), ("base test", "P@20")]]
points.columns = ["alarms", "precision"]
frontier = points.loc[[NAMES[m] for m in ["dronetrack-counter", "dronetrack", "yolo-bytetrack", "dronetrack-cenek-weights", "motion-kf"]]]
label_offsets = {
    "motion-kf": (-6, 5, "right"),
    "yolo-bytetrack": (6, 3, "left"),
    "dronetrack": (6, -9, "left"),
    "dronetrack-no-motion": (6, -9, "left"),
    "dronetrack-counter": (6, 0, "left"),
    "dronetrack-cenek-weights": (-6, 5, "right"),
}

fig, ax = plt.subplots(figsize=(4.2, 2.8))
ax.plot(frontier.alarms, frontier.precision, color=GREY, lw=1.2, zorder=1)
for method, (dx, dy, ha) in label_offsets.items():
    alarms_per_min, precision = points.loc[NAMES[method]]
    ours = method.startswith("dronetrack")
    ax.scatter(alarms_per_min, precision, s=50 if method == "dronetrack" else 28, color=BLUE if ours else ORANGE,
               marker="s" if ours and method != "dronetrack" else "o", edgecolor="white", lw=1, zorder=3)
    label = NAMES[method] + (" (proposed)" if method == "dronetrack" else "")
    ax.annotate(label, (alarms_per_min, precision), xytext=(dx, dy), textcoords="offset points", ha=ha, fontsize=7.5)
ax.set(xscale="log", xlim=(0.6, 60), ylim=(0.68, 0.94), xlabel="False alarms per minute (Halmstad, drone-free)",
       ylabel="P@20 (base test flight)")
ax.set_xticks([1, 3, 10, 30], ["1", "3", "10", "30"])
ax.grid(color=GREY, lw=0.5)
fig.savefig(FIGURES / "tradeoff.pdf", dpi=300)

## 6. Tracking at work

In [22]:
test = {s.name: s for s in datasets.load(config, "cenek", "test")}
camera = test["dataset4/cam4"]
frames = with_labels(outputs("dronetrack", "dataset4/cam4"), camera)

# the 8-second window with the longest path in which every frame is a hit and the drone stays clear of the border
width, height = 1920, 1080
frames["inside"] = frames.x.between(80, width - 80) & frames.y.between(80, height - 80)
frames["moved"] = np.hypot(frames.x.diff(), frames.y.diff())
length = round(8 * len(frames) / (frames.t.iloc[-1] - frames.t.iloc[0]))
rolling = frames[["hit", "inside", "moved"]].astype(float).rolling(length)
windows = pd.DataFrame({"hits": rolling.hit.mean(), "inside": rolling.inside.min(), "path": rolling.moved.sum()})
end = windows[(windows.hits >= 0.98) & (windows.inside == 1)].path.idxmax()
start = frames.index[frames.index.get_loc(end) - length + 1]
window = frames.loc[start:end]
print(f"frames {start}-{end}: {window.hit.mean():.0%} hits, path {windows.path[end]:.0f} px, "
      f"box {window.w.median():.0f} x {window.h.median():.0f} px")

frames 8650-8889: 100% hits, path 1468 px, box 20 x 16 px


In [23]:
snapshots = [start, window.index[len(window) // 2], end]
images = {f.index: f.image[:, :, ::-1] for f in camera.frames(select=lambda i: i in snapshots)}

fig = plt.figure(figsize=(6.4, 3.0))
grid = fig.add_gridspec(3, 2, width_ratios=[5.3, 1], wspace=0.03, hspace=0.06)
ax = fig.add_subplot(grid[:, 0])
ax.imshow((images[end] * 0.85).astype(np.uint8))
track = window[window.declared][["x", "y"]].to_numpy()
segments = np.stack([track[:-1], track[1:]], axis=1)
line = LineCollection(segments, cmap="Blues", norm=plt.Normalize(-0.3 * len(segments), len(segments)), lw=2)
line.set_array(np.arange(len(segments)))
ax.add_collection(line)
for letter, frame in zip("abc", snapshots):
    x, y = window.loc[frame, ["x", "y"]]
    ax.plot(x, y, "o", ms=11, mfc="none", mec="white", mew=1.2)
    ax.text(x - 22, y - 22, letter, color="white", ha="right", va="bottom", fontsize=9)
ax.set_axis_off()

half = 36
for row, (letter, frame) in enumerate(zip("abc", snapshots)):
    x, y, w, h, x_label, y_label = window.loc[frame, ["x", "y", "w", "h", "x_label", "y_label"]]
    left, top = int(x) - half, int(y) - half
    ax = fig.add_subplot(grid[row, 1])
    ax.imshow(images[frame][top : top + 2 * half, left : left + 2 * half], interpolation="nearest")
    ax.add_patch(Rectangle((x - w / 2 - left, y - h / 2 - top), w, h, fill=False, ec=BLUE, lw=1.3))
    ax.plot(x_label - left, y_label - top, "+", color=GREEN, ms=7, mew=1.2)
    ax.text(3, 3, f"{letter}   {window.t[frame] - window.t[start]:.0f} s", color="white", va="top", fontsize=7.5)
    ax.set_axis_off()

fig.savefig(FIGURES / "tracking.pdf", dpi=300)
fig.savefig("report/hero.png", dpi=300, bbox_inches="tight")

## 7. Failure analysis

### 7.1 Static false target, camera 6

In [24]:
cam6 = outputs("dronetrack", "dataset4/cam6")
declared = cam6[cam6.declared]
most_frequent = declared[["x", "y"]].round(-1).value_counts().idxmax()
locked = declared[np.hypot(declared.x - most_frequent[0], declared.y - most_frequent[1]) < 30]
box = locked[["x", "y", "w", "h"]].median()
print(f"held near ({box.x:.0f}, {box.y:.0f}), {box.w:.0f} x {box.h:.0f} px, in {len(locked)} of {len(cam6)} frames "
      f"({len(locked) / len(cam6):.0%})")

held near (138, 786), 57 x 25 px, in 9267 of 11292 frames (82%)


### 7.2 Small drone, camera 5

In [25]:
cam5 = test["dataset4/cam5"]
visible = cam5.drone_boxes().set_index("frame")
sampled = set(visible.index[np.linspace(0, len(visible) - 1, 60).astype(int)])

sizes, cam5_crops = [], []
for frame in cam5.frames(select=lambda i: i in sampled):
    x, y = visible.loc[frame.index, ["x", "y"]]
    (_, _, w, h), ok = box_from_point(cv2.cvtColor(frame.image, cv2.COLOR_BGR2GRAY), x, y)
    if ok:
        sizes.append(max(w, h) - 4)
    cam5_crops.append(frame.image[int(y) - 60 : int(y) + 60, int(x) - 60 : int(x) + 60, ::-1])

low, high = np.percentile(sizes, [25, 75])
print(f"camera 5 drone, longer side: {low:.0f}-{high:.0f} px (interquartile, {len(sizes)} of {len(sampled)} frames segmented)")

camera 5 drone, longer side: 5-12 px (interquartile, 40 of 60 frames segmented)


### 7.3 Low-confidence detections and night, FRED

In [26]:
fred = {s.name.split("/")[-1]: s for s in datasets.load(config, "fred", "test")}
brightness = {}
for name, sequence in fred.items():
    sample = sequence.images[:: len(sequence.images) // 10]
    brightness[name] = np.mean([cv2.imread(str(p), cv2.IMREAD_GRAYSCALE).mean() for p in sample])
pd.Series(brightness, name="mean grey level").to_frame().T

,12,60,63,64,65,80,93,110,111,115,219
mean grey level,95.426,81.712,83.655,81.806,87.184,73.6,83.382,69.164,9.037,71.872,101.138


In [27]:
models = {DETECTORS[m]: YOLO(config.weights(m, "pt")) for m in DETECTORS}


def iou(a, b):
    w = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    h = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    return w * h / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - w * h)


def crop_around(sequence, label, size=640):
    """640 x 640 crop around a label, as the pipeline would cut it, and the label box in crop coordinates"""
    image = cv2.imread(str(sequence.images[int(label.frame)]))
    x0 = int(np.clip(label.x - size / 2, 0, image.shape[1] - size))
    y0 = int(np.clip(label.y - size / 2, 0, max(image.shape[0] - size, 0)))
    tile = np.zeros((size, size, 3), np.uint8)
    part = image[y0 : y0 + size, x0 : x0 + size]
    tile[: part.shape[0], : part.shape[1]] = part
    truth = (label.x - label.w / 2 - x0, label.y - label.h / 2 - y0, label.x + label.w / 2 - x0, label.y + label.h / 2 - y0)
    return tile, truth


def best_match(model, tile, truth):
    """Most confident drone detection overlapping the label, as (confidence, IoU, box)"""
    result = model.predict(tile, imgsz=640, conf=0.05, verbose=False)[0]
    matches = []
    for box, confidence, cls in zip(result.boxes.xyxy.tolist(), result.boxes.conf.tolist(), result.boxes.cls.tolist()):
        if cls == 0 and iou(truth, box) > 0.1:
            matches.append((confidence, iou(truth, box), box))
    return max(matches, default=(np.nan, np.nan, None), key=lambda m: m[0])

In [28]:
rows = []
for name in ["12", "60", "80", "111", "115"]:
    labels = fred[name].drone_boxes()
    picks = np.linspace(0.1 * len(labels), 0.9 * len(labels), 6).astype(int)
    for _, label in labels.iloc[picks].iterrows():
        tile, truth = crop_around(fred[name], label)
        for detector, model in models.items():
            confidence, overlap, _ = best_match(model, tile, truth)
            rows.append({"sequence": name, "frame": int(label.frame), "detector": detector,
                         "confidence": confidence, "iou": overlap})

crops = pd.DataFrame(rows)
crops.groupby(["detector", "sequence"]).agg(
    found=("confidence", "count"),
    crops=("confidence", "size"),
    confidence_min=("confidence", "min"),
    confidence_max=("confidence", "max"),
    iou_min=("iou", "min"),
    iou_max=("iou", "max"),
)

found  crops  confidence_min  confidence_max  iou_min  iou_max
detector  sequence                                                                
base only 111           0      6             NaN             NaN      NaN      NaN
          115           3      6           0.713           0.807    0.150    0.174
          12            3      6           0.645           0.829    0.120    0.174
          60            5      6           0.703           0.841    0.209    0.299
          80            2      6           0.171           0.839    0.115    0.136
mixed     111           0      6             NaN             NaN      NaN      NaN
          115           4      6           0.121           0.155    0.272    0.718
          12            4      6           0.110           0.244    0.493    0.627
          60            6      6           0.182           0.558    0.141    0.716
          80            4      6           0.133           0.197    0.317    0.722

In [29]:
failing = crops[(crops.detector == "mixed") & crops.sequence.isin(["12", "80", "115"])]
print(f"mixed detector, failing daylight sequences: found in {failing.confidence.notna().sum()} of {len(failing)} crops, "
      f"confidence {failing.confidence.min():.2f}-{failing.confidence.max():.2f} "
      f"(threshold {config.pipeline.detector.conf}), IoU {failing.iou.min():.2f}-{failing.iou.max():.2f}")

mixed detector, failing daylight sequences: found in 12 of 18 crops, confidence 0.11-0.24 (threshold 0.25), IoU 0.27-0.72


### 7.4 Figure

In [30]:
def outline(ax, x0, y0, x1, y1, colour, **style):
    ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec=colour, **style))


def fred_view(name, frame, half=110):
    """Crop around a FRED label, the label box and the best mixed-detector match, in view coordinates"""
    label = fred[name].annotations.query("frame == @frame").iloc[0]
    tile, truth = crop_around(fred[name], label)
    confidence, _, detection = best_match(models["mixed"], tile, truth)
    x0 = int(max((truth[0] + truth[2]) / 2 - half, 0))
    y0 = int(max((truth[1] + truth[3]) / 2 - half, 0))
    shift = lambda b: (b[0] - x0, b[1] - y0, b[2] - x0, b[3] - y0)
    return tile[y0 : y0 + 2 * half, x0 : x0 + 2 * half, ::-1], shift(truth), detection and shift(detection), confidence


cam6_frame = int(locked.index[len(locked) // 2])
cam6_image = next(test["dataset4/cam6"].frames(select=lambda i: i == cam6_frame)).image[:, :, ::-1]
low_confidence = crops.query("detector == 'mixed' and sequence == '80'").dropna().sort_values("iou").frame.iloc[-1]
night = fred["111"].drone_boxes().frame.iloc[len(fred["111"].drone_boxes()) // 2]

fig = plt.figure(figsize=(6.4, 4.3))
grid = fig.add_gridspec(2, 3, hspace=0.25, wspace=0.05)
panels = [fig.add_subplot(grid[0, :2]), fig.add_subplot(grid[0, 2])] + [fig.add_subplot(grid[1, i]) for i in range(3)]

panels[0].imshow(cam6_image)
outline(panels[0], box.x - box.w / 2, box.y - box.h / 2, box.x + box.w / 2, box.y + box.h / 2, ORANGE, lw=1.5)
panels[0].annotate("", xy=(box.x, box.y - box.h), xytext=(box.x + 250, box.y - 200),
                   arrowprops=dict(arrowstyle="->", color=ORANGE, lw=1.5))
panels[0].set_title("(a) Camera 6: static false target", loc="left", fontsize=8.5)

panels[1].imshow(cam6_image[int(box.y) - 90 : int(box.y) + 90, max(int(box.x) - 90, 0) : int(box.x) + 160])
panels[1].set_title("(b) Close-up: cabinet lid", loc="left", fontsize=8.5)

panels[2].imshow(cam5_crops[len(cam5_crops) // 2])
panels[2].set_title("(c) Camera 5: small drone", loc="left", fontsize=8.5)

view, truth, detected, confidence = fred_view("80", low_confidence)
panels[3].imshow(view)
outline(panels[3], *detected, ORANGE, lw=1.4)
outline(panels[3], *truth, GREEN, lw=1.2, ls="--")
panels[3].set_title(f"(d) FRED 80: confidence {confidence:.2f}", loc="left", fontsize=8.5)

view, truth, _, _ = fred_view("111", night)
panels[4].imshow(np.clip(view.mean(axis=2) * 4, 0, 255), cmap="gray", vmin=0, vmax=255)
outline(panels[4], *truth, GREEN, lw=1.2, ls="--")
panels[4].set_title("(e) FRED 111: night", loc="left", fontsize=8.5)

for ax in panels:
    ax.set_axis_off()
fig.savefig(FIGURES / "failures.pdf", dpi=300)

## 8. Robustness

In [31]:
conditions = ["clean", "downscale-2", "downscale-4", "downscale-8", "low-light", "noise", "blur"]
robustness = pd.concat(
    {c: per_sequence("dronetrack", "cenek-test-subset", c, suite="robustness").set_index("sequence") for c in conditions},
    names=["condition"],
)
precision = robustness.precision_20.unstack().loc[conditions]
precision["mean cam1-2"] = precision[["dataset4/cam1", "dataset4/cam2"]].mean(axis=1)
pd.concat({"P@20": precision, "SA": robustness.state_accuracy.unstack().loc[conditions]}, axis=1)

P@20                                                    SA                            
sequence    dataset4/cam1 dataset4/cam2 dataset4/cam6 mean cam1-2 dataset4/cam1 dataset4/cam2 dataset4/cam6
condition                                                                                                  
clean               0.924         0.981         0.395       0.953         0.590         0.880     1.712e-01
downscale-2         0.851         0.892         0.000       0.872         0.668         0.916     8.856e-05
downscale-4         0.255         0.346         0.049       0.300         0.781         0.705     6.759e-01
downscale-8         0.006         0.020         0.000       0.013         0.733         0.563     6.603e-01
low-light           0.682         0.446         0.390       0.564         0.809         0.252     7.875e-01
noise               0.968         0.852         0.170       0.910         0.900         0.513     7.802e-02
blur                0.640         0.600         0.124       0.620         0.701         0.816     5.487e-01

In [32]:
ticks = ["Clean", r"Range $\times 2$", r"Range $\times 4$", r"Range $\times 8$", "Low light", "Noise", "Blur"]
x, width = np.arange(len(conditions)), 0.36

fig, ax = plt.subplots(figsize=(6.4, 2.2))
for offset, sequence, label, colour in [(-0.5, "dataset4/cam1", "Camera 1", BLUE), (0.5, "dataset4/cam2", "Camera 2", ORANGE)]:
    ax.bar(x + offset * width, precision[sequence], width - 0.04, color=colour, label=label)
ax.set_xticks(x, ticks)
ax.set(ylim=(0, 1), ylabel="P@20")
ax.grid(axis="y", color=GREY, lw=0.5)
ax.set_axisbelow(True)
ax.legend(ncol=2, loc="lower center", bbox_to_anchor=(0.5, 1.0))
fig.savefig(FIGURES / "robustness.pdf", dpi=300)

## 9. Real time

In [33]:
live = load_metrics(RUNS).query("suite == 'realtime'").iloc[0]
offline = metrics.query("method == 'dronetrack' and test_set == 'cenek-test'").iloc[0]
rows = ["precision_10", "precision_20", "state_accuracy", "latency_p50_ms", "latency_p95_ms", "stale_fraction"]
pd.DataFrame({"offline": offline, "live replay": live}).loc[rows].astype(float)

,offline,live replay
precision_10,0.794,0.799
precision_20,0.798,0.806
state_accuracy,0.733,0.735
latency_p50_ms,33.868,35.021
latency_p95_ms,42.862,43.871
stale_fraction,NaN,0.152


In [34]:
pd.read_csv(RUNS / "benchmark/latency.csv")

,clip,configuration,frames,p50_ms,p95_ms,p99_ms,decode_ms,motion_ms,detector_ms,tracker_ms,realtime_factor
0,1080p,dronetrack / PyTorch / auto,1450,40.97,45.84,47.30,1.04,4.50,33.45,0.08,0.86
1,1080p,dronetrack / ONNX Runtime / CPU,1450,295.62,299.95,307.09,1.28,6.68,282.78,0.08,0.12
2,1080p,motion-kf / CPU,1450,5.63,6.31,6.95,1.03,4.59,0.00,0.07,5.89
3,2160p,dronetrack / PyTorch / auto,1450,31.52,45.97,48.90,2.60,5.67,26.02,0.11,1.37
4,2160p,dronetrack / ONNX Runtime / CPU,1450,280.09,302.40,314.05,6.04,7.32,270.75,0.12,0.17
5,2160p,motion-kf / CPU,1450,8.22,9.36,10.24,2.68,5.58,0.00,0.07,5.66


In [35]:
started = pd.Timestamp((RUNS / "run.pid").stat().st_mtime, unit="s")
finished = pd.Timestamp(max(p.stat().st_mtime for p in (RUNS / "logs").glob("*.log")), unit="s")
print(f"full study: {(finished - started).total_seconds() / 3600:.0f} hours")

full study: 28 hours
